<a href="https://colab.research.google.com/github/NataliaPoluektova/MyFirstProject/blob/main/Lab_CCSI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Лабораторна робота №2.** Оптимізація обчислювальних графів: Від архітектурних патернів до стиснення моделей у Google Colab

**Мета роботи:**  На практиці освоїти життєвий цикл оптимізації ШІ: навчитися аналізувати обчислювальну складність графа, застосовувати алгоритми стиснення (квантування) та виконувати операторне злиття через графову компіляцію в середовищі Google Colab.

Перед тим як писати код, розберемо три головні інженерні поняття нашого модуля, які визначають швидкість ШІ на реальному залізі:

**Обчислювальний граф (DAG):**  Нейромережа в пам’яті комп'ютера — це не просто формули, а орієнтований ациклічний граф. Його вузли — це математичні оператори (Mul, Add, ReLU), а ребра — тензори (масиви даних).

**Квадрат компромісів (Trade-offs):** Інженерна розробка ШІ — це завжди пошук балансу. Ми не можемо безмежно стискати модель без ризику втратити її точність (Accuracy).

Наше завдання — радикально зменшити затримку (Latency) та об'єм пам'яті (VRAM/RAM), мінімально зачепивши якість прогнозів.

**Метрики заліза (MAC та FLOP):** Кожен прохід даних крізь граф коштує процесору тактів роботи. Ми вимірюємо цю складність у FLOP (окремі плаваючі операції) та MAC (помножити-додати за один такт). Пам'ятаємо: 1 MAC = 2 FLOP.

**Крок 1.** Підготовка інструментарію та профайлера графа
Оскільки ми працюємо як інженери, нам потрібен точний вимірювальний прилад — профайлер. Ми встановимо бібліотеку fvcore (розробка Meta AI), яка вміє аналізувати граф моделі та підраховувати кожну математичну операцію «під капотом».

In [ ]:
!pip install fvcore --quiet
print(" Профайлер графа fvcore успішно встановлено!")

**Крок 2.** Проектування архітектури та аналіз її складності
Створимо базову нейромережу. Вона складається з двох ключових блоків:

Згортковий шар (Conv2d) — аналізує просторові ознаки картинки за допомогою сканування невеликим ядром.

Повнозв'язний шар (Linear/Dense) — агрегує всі знайдені ознаки докупи та приймає фінальне рішення (видає 10 цифр-ймовірностей для класів).

In [ ]:
import torch
import torch.nn as nn
import os
from fvcore.nn import FlopCountAnalysis

class CoreResearchNet(nn.Module):
    def __init__(self):
        super(CoreResearchNet, self).__init__()
        # Вхід: 3 канали (RGB), вихід: 32 канали, ядро фільтра: 3x3
        self.conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU() # Активація, що занулює негативні значення
        # Повнозв'язний шар: з'єднує всі 32 канали картинки розміром 32x32 з 10 виходами
        self.fc = nn.Linear(32 * 32 * 32, 10)

    def forward(self, x):
        # Описуємо рух тензорів по ребрах нашого графа (Forward Pass)
        x = self.relu(self.conv(x))
        x = x.view(x.size(0), -1) # Вирівнюємо матрицю в один довгий вектор
        x = self.fc(x)
        return x

# 1. Створюємо екземпляр моделі в режимі тестування
baseline_model = CoreResearchNet().eval()

# 2. Генеруємо фейкове зображення (1 картинка, 3 канали RGB, розмір 32x32 пікселі)
dummy_image = torch.randn(1, 3, 32, 32)

# 3. Фіксуємо базову вагу моделі на диску (формат високої точності FP32 - 32 біти на число)
torch.save(baseline_model.state_dict(), "baseline_fp32.pth")
fp32_size = os.path.getsize('baseline_fp32.pth') / 1024
print(f" Вага базової моделі (FP32): {fp32_size:.2f} KB")

# 4. Запускаємо профайлер для аналізу структури DAG
graph_analyzer = FlopCountAnalysis(baseline_model, dummy_image)
total_flops = graph_analyzer.total()
total_macs = total_flops // 2 # Залізо виконує множення і додавання в один такт

print(f" Математичний аудит графа нейромережі:")
print(f"  ├─ Кількість операцій FLOP: {total_flops:,}")
print(f"  └─ Кількість тактів заліза (MAC): {total_macs:,}")

**Крок 3.** Інженерне стиснення: Пост-тренувальне квантування (PTQ)Тепер застосуємо алгоритм Post-Training Quantization (PTQ). Наше завдання — перевести ваги моделі з дробового формату FP32 ($32\text{ біти}$) у компактний цілочисельний INT8 ($8\text{ біт}$).Для цього фреймворк проганяє крізь модель невелику порцію даних (калібрування), фіксує максимальні та мінімальні амплітуди сигналів і вираховує два параметри: масштаб ($S$) та нульову точку ($Z$).

In [ ]:
import torch.quantization

model_to_compress = CoreResearchNet().eval()

# Налаштовуємо конвеєр під інструкції процесорів архітектури Colab (x86)
model_to_compress.qconfig = torch.quantization.get_default_qconfig('x86')

# Крок А: Підготовка графа (вставка віртуальних спостерігачів за числами)
prepared_model = torch.quantization.prepare(model_to_compress, inplace=False)

# Крок Б: Калібрування. Пропускаємо 20 випадкових картинок, щоб граф дізнався свої S та Z
for _ in range(20):
    prepared_model(torch.randn(1, 3, 64, 64))

# Крок В: Конвертація. Фізичне стиснення - заміна 32-бітних дробів на 8-бітні цілі числа
quantized_model = torch.quantization.convert(prepared_model, inplace=False)

# Зберігаємо результат для порівняння ваги
torch.save(quantized_model.state_dict(), "compressed_int8.pth")
int8_size = os.path.getsize('compressed_int8.pth') / 1024

print(f" Процес PTQ квантування завершено!")
print(f" Вага стиснутої моделі (INT8): {int8_size:.2f} KB")
print(f" Фізичне зменшення об'єму пам'яті в: {fp32_size / int8_size:.2f} разів!")

**Крок 4. **Графова компіляція: Злиття операторів (Operator Fusion)
Зараз модель стиснута, але процесор все одно виконує її крок за кроком, витрачаючи час на пересилання проміжних тензорів туди-сюди.

Ми активуємо сучасний графовий компілятор torch.compile (з філософії PyTorch 2.x). Він проаналізує наш DAG, знайде суміжні операції Conv + ReLU і фізично сплавить їх в один монолітний вузол коду. Процесор виконає їх всередині своїх швидких регістрів за один прохід.

In [ ]:
import time

num_tests = 300 # Кількість прогонів для точності статистики

# 1. Замір швидкості базової архітектури без компіляції
start = time.time()
with torch.no_grad():
    for _ in range(num_tests):
        _ = baseline_model(dummy_image)
raw_latency = (time.time() - start) / num_tests

print(" Графовий компілятор аналізує DAG та зшиває оператори (Operator Fusion)...")
# 2. Запуск графового компілятора
compiled_model = torch.compile(baseline_model)

# Прогрів (Warm-up): перший запуск ініціює компіляцію, він триває кілька секунд
with torch.no_grad():
    _ = compiled_model(dummy_image)

# 3. Замір швидкості скомпільованого машинного коду
start = time.time()
with torch.no_grad():
    for _ in range(num_tests):
        _ = compiled_model(dummy_image)
compiled_latency = (time.time() - start) / num_tests

# Фінальний інженерний аналіз результатів
print("\n ЗВІТ З АПАРАТНОЇ ОПТИМІЗАЦІЇ:")
print(f"  ├─ Затримка базової моделі (Raw Latency FP32): {raw_latency * 1000:.4f} мс")
print(f"  ├─ Затримка після компіляції (Fused Latency):  {compiled_latency * 1000:.4f} мс")
print(f"  └─  Чисте прискорення заліза за рахунок Fusion: {raw_latency / compiled_latency:.2f}х")

Проаналізуйте результат (чи бачите Ви парадокс, чим він може бути пояснений?):


**Крок 5.** Оскільки перша модель виявилася занадто легкою і викликала оверхед, ми проаналізуємо роботу мережи з 8 шарів.

In [ ]:
print(" Створюємо DeepProductionNet (8 згорткових шарів)...")

class DeepProductionNet(nn.Module):
    def __init__(self):
        super(DeepProductionNet, self).__init__()
        # Послідовний ланцюжок шарів для створення великого і важкого графа
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),  nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU()
        )
        self.fc = nn.Linear(128 * 32 * 32, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Ініціалізація та замір швидкості
deep_model = DeepProductionNet().eval()

# 1. Замір базової швидкості важкої моделі
start = time.time()
with torch.no_grad():
    for _ in range(num_tests):
        _ = deep_model(dummy_image)
deep_raw_latency = (time.time() - start) / num_tests

# 2. Компіляція великого графа
print(" Тепер граф великий. Компілятор оптимізує масштабну структуру...")
compiled_deep_model = torch.compile(deep_model)
with torch.no_grad():
    _ = compiled_deep_model(dummy_image) # Розігрів

# 3. Замір швидкості після оптимізації
start = time.time()
with torch.no_grad():
    for _ in range(num_tests):
        _ = compiled_deep_model(dummy_image)
deep_compiled_latency = (time.time() - start) / num_tests

print("\n ЗВІТ З ОПТИМІЗАЦІЇ ВАЖКОЇ МОДЕЛІ:")
print(f"  ├─ Затримка без компіляції (Raw Latency): {deep_raw_latency * 1000:.4f} мс")
print(f"  ├─ Затримка після компіляції (Fused Latency): {deep_compiled_latency * 1000:.4f} мс")
print(f"  └─  Чисте прискорення заліза: {deep_raw_latency / deep_compiled_latency:.2f}х")

**Завдання для самостійного виконання 1.** Експеримент зі зміною розмірності (Ефект масштабу)
Змініть розмір вхідного фейкового зображення dummy_image. Замість стандартного для датасету CIFAR розміру 32x32, подайте на вхід картинку більшої роздільної здатності — 64x64.

Зробіть це і для моделі з кроку 2.

In [ ]:
#Ваш код:


Ось ми й зловили справжню класичну помилку інженерії нейромереж!
Цей RuntimeError пов'язаний з базовим інструментом профайлера fvcore, який працює через механізм трасування графа (JIT Tracing).Давайте детально розберемо, що саме зламалося та як цей збій перетворити на ще один потужний навчальний інструмент для студентів.

 Анатомія помилки: Чому граф «вибухнув»?

 Помилка каже:RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x131072 and 32768x10)Це означає, що на повнозв'язний шар Linear (матриця mat2 розміром $32768 \times 10$) прилетів тензор (mat1) розміром $131072$. Вони математично не сумісні для множення матриць.

 Замість того, щоб вручну підбирати магічні числа в nn.Linear(32 * 32 * 32, 10), в індустрії використовують шар nn.AdaptiveAvgPool2d (адаптивний пулінг) або динамічне визначення розміру.

Давайте оновимо код нашої першої моделі в Клітинці 2 так, щоб вона була абсолютно стійкою до трасування компілятором і профайлером, незалежно від версії Python:



In [ ]:
import torch
import torch.nn as nn
import os
from fvcore.nn import FlopCountAnalysis

class CoreResearchNet(nn.Module):
    def __init__(self):
        super(CoreResearchNet, self).__init__()
        # Вхід: 3 канали (RGB), вихід: 32 канали, ядро фільтра: 3x3
        self.conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # ДОДАЄМО: Адаптивний пулінг, який примусово стискає будь-яку картинку
        # до фіксованого просторового розміру 32x32 перед лінійним шаром.
        # Це гарантує стабільність графа при будь-яких бачах та трасувальниках!
        self.pool = nn.AdaptiveAvgPool2d((32, 32))

        # Тепер вхід чітко дорівнює 32 * 32 * 32 = 32768
        self.fc = nn.Linear(32 * 32 * 32, 10)

    def forward(self, x):
        x = self.relu(self.conv(x))
        x = self.pool(x) # Стабілізуємо розмірність тензора в графі
        x = x.view(x.size(0), -1) # Вирівнюємо в один вектор
        x = self.fc(x)
        return x

# 1. Створюємо екземпляр моделі в режимі тестування
baseline_model = CoreResearchNet().eval()

# 2. Генеруємо фейкове зображення (1 картинка, 3 канали RGB, розмір 32x32 пікселі)
dummy_image = torch.randn(1, 3, 32, 32)

# 3. Фіксуємо базову вагу моделі на диску
torch.save(baseline_model.state_dict(), "baseline_fp32.pth")
fp32_size = os.path.getsize('baseline_fp32.pth') / 1024
print(f" Вага базової моделі (FP32): {fp32_size:.2f} KB")

# 4. Запускаємо профайлер — тепер граф пройде трасування без помилок!
graph_analyzer = FlopCountAnalysis(baseline_model, dummy_image)
total_flops = graph_analyzer.total()
total_macs = total_flops // 2

print(f"\n Математичний аудит графа нейромережі:")
print(f"  ├─ Кількість операцій FLOP: {total_flops:,}")
print(f"  └─ Кількість тактів заліза (MAC): {total_macs:,}")

 У скільки разів збільшилася кількість операцій FLOP для моделей? Чому?



**Завдання для самостійного виконання 2.** Математичний розрахунок вручнуВикористовуючи формулу з лекцій для повнозв'язного шару:$$\text{Кількість MAC} = N \cdot M$$Де $N$ — кількість вхідних елементів, а $M$ — кількість вихідних нейронів.

Розрахуйте вручну, скільки операцій MAC та FLOP виконує шар self.fc у нашій базовій моделі для початкового варіанту (картинка $32 \times 32$), якщо на вхід йому приходить вектор розміром $N = 32768$ ($32 \times 32 \times 32$), а на виході ми маємо $M = 10$ класів. Порівняйте свій ручний результат із тим, що видав профайлер fvcore у консолі.

Ваші результати та висновки:

